# WORK IN PROGRESS

In [0]:
# Reverse Geocoder
# Author: Steve Scott (sscott1@cityops.nyc.gov)
# Created: 2026-03-17
#
# Description:
# Takes the blockface midpoints from blockfaces.geojson and joins them to
# property addresses in pad.zip by finding the nearest point in pad.zip
# to each blockface midpoint.


# Parse address into house_number and street, and output with borough
number_street_borough = filtered[['address', 'borough']].copy()
number_street_borough['house_number'] = number_street_borough['address'].str.split().str[0]
# if the house number is a number with or without hyphens, then it is a valid house number.
# if it is invalid, then the house number is set to NaN and the street name is the entire address.

valid_house_number = number_street_borough['house_number'].str.match(r'^\d+(-\d+)?$').fillna(False)
number_street_borough['house_number'] = number_street_borough['house_number'].where(valid_house_number.values)

boro_mapping = {
    'MN' : 'Manhattan',
    'BX' : 'Bronx',
    'BK' : 'Brooklyn',
    'QN' : 'Queens',
    'SI' : 'Staten Island'
}

number_street_borough['borough'] = number_street_borough['borough'].map(boro_mapping)

number_street_borough['street'] = number_street_borough.apply(lambda row: row['address'].split(' ', 1)[1] if pd.notna(row['house_number']) else row['address'], axis=1)
number_street_borough = number_street_borough[['house_number', 'street', 'borough']]
number_street_borough.to_csv('number_street_borough.csv', index=False)

In [0]:
from pyspark.sql.functions import expr
import os

In [0]:
blockface_midpoints = spark.table("scorecard_fulcrum.geo.pavement_edge_midpoints")

read pluto dataset

In [0]:
pluto = spark.table('scorecard_fulcrum.geo.pluto_2026')

pluto = pluto.withColumn(
    "geometry",
    expr("ST_Point(cast(longitude as double), cast(latitude as double))")
)


Transform to state plane because you are doing distance calculations

In [0]:
blockface_midpoints_proj = blockface_midpoints.withColumn(
    "geometry",
    expr("ST_Transform(ST_SetSRID(geometry, 4326), 2263)")
)

pluto_proj = pluto.withColumn(
    "geometry",
    expr("ST_Transform(ST_SetSRID(geometry, 4326), 2263)")
)

Spatial Join


Cell 9 performs a nearest-neighbor spatial join between blockface midpoints and PLUTO tax lot points. Here's the breakdown:

Register temp views — blockface_midpoints_proj and pluto_proj are exposed as SQL-queryable views (bmp_view, pluto_view) so we can use Spark SQL spatial functions.

Spatial filter (JOIN ... ON ST_Intersects(ST_Buffer(b.geometry, 500), p.geometry)) — For each midpoint, creates a 500-foot search radius (EPSG:2263 uses US feet) and only considers PLUTO points that fall within it. This avoids a full cross join and keeps the computation tractable.

Ranking (ROW_NUMBER() OVER (PARTITION BY b.TARGET_FID ORDER BY ST_Distance(...))) — Among the candidate PLUTO points within 500 feet, ranks them by actual distance to the midpoint. Each midpoint (TARGET_FID) gets its own ranking.

Pick nearest (WHERE rn = 1) — Keeps only the closest PLUTO record per midpoint.

Clean up (.drop("rn")) — Removes the helper ranking column so downstream cells aren't affected.

The result is one row per blockface midpoint, enriched with address, borough, latitude, and longitude from the nearest PLUTO property.

In [0]:
blockface_midpoints_proj.createOrReplaceTempView("bmp_view")
pluto_proj.createOrReplaceTempView("pluto_view")

joined = spark.sql("""
    SELECT * FROM (
        SELECT b.*, p.address, p.borough, p.latitude, p.longitude,
               ROW_NUMBER() OVER (PARTITION BY b.TARGET_FID ORDER BY ST_Distance(b.geometry, p.geometry)) as rn
        FROM bmp_view b
        JOIN pluto_view p ON ST_Intersects(ST_Buffer(b.geometry, 500), p.geometry)
    ) WHERE rn = 1
""").drop("rn")

retransform back to WGC

In [0]:
joined = joined.withColumn(
    "geometry",
    expr("ST_Transform(ST_SetSRID(geometry, 2263), 4326)")
)

Filter the results

In [0]:
filtered = joined.select(
    "latitude", "longitude", "geometry", "address", "borough", "TARGET_FID", "blockf_id",
    "conflated", "feat_code", "shape_leng", "source_id", "status", "sub_code", "district",
    "fid_1", "globalid", "section", "sectioncod", "districtco", "Shape_Le_1", "MIDX", "MIDY"
)

filtered.write.saveAsTable('scorecard_fulcrum.geo.midpoints_with_address')